In [ ]:
# ================================================================
# 1) USER CONTROL — CHANGE ONLY RUN_SEED FOR THE FIVE PUBLICATION SEEDS
# ================================================================
from pathlib import Path

RUN_SEED = 42  # one of: 42, 1337, 2026, 31415, 271828
RESUME_FROM_INPUT_PROGRESS = True  # reuse exact same-seed finalized FacetGate progress mounted under /kaggle/input

PUBLICATION_SEEDS = [42, 1337, 2026, 31415, 271828]
if RUN_SEED not in PUBLICATION_SEEDS:
    raise ValueError(f"RUN_SEED must be one of {PUBLICATION_SEEDS}; got {RUN_SEED}")

MODEL_ID = "openbmb/MiniCPM-V-4_5"
MODEL_REVISION = "daef484c35ec93210ec93c5e901f8f3e9b78ee34"
ARCHITECTURE_NAME = "FacetGate"
PROTOCOL_VERSION = "1.0.1"
PROTOCOL_SHA256 = "37b7d30a994fb98225d49d76cca5768a838d456abf844dc1a5d93019a0cd15aa"
PARENT_GATED_PROTOCOL_SHA256 = "627bdf5a77ab694581ff590c32f8885c901ebad09e8df86dbab07762caaf95dd"
MINICPM_NATIVE_BINDING_SHA256 = "7b48b0a1abe63fc3c1aa404a96cc362fb1ae3480605bcbbcc1f0098d43d9d32f"

# Certified validation-selected MiniCPM Frozen Common Gated v1.1 parents.
# These are provenance checks only; if no same-seed parent is mounted the protocol
# still selects SIMULTANEOUS_FROM_SCRATCH.
CERTIFIED_PARENT = {
    42:     {"step":932, "checkpoint_sha256":"ffd8ae90ce0130012fd9a99fe28225919af92eedfbd0dc03d0b982789996e2db"},
    1337:   {"step":800, "checkpoint_sha256":"53304788ffc7bb928c303c88cd1ee29d1a8aa868dc85d0d680109bf9e6d2371c"},
    2026:   {"step":800, "checkpoint_sha256":"45bb11afc5064baf325c7a24f83964e2e4b78e4c82dc0fc8e2001e69afd1036f"},
    31415:  {"step":800, "checkpoint_sha256":"651564574b63936786746b67fb120c4036a6829b506a10d74abaa679ad313759"},
    271828: {"step":800, "checkpoint_sha256":"943b832d1d7884b4d2d45a8a0bd2d4070d59f9bab8a7206007404324aff8e11f"},
}
CERTIFIED_PARENT_ADAPTER_SERIALIZATION = "norm_fc1_fc2"
CERTIFIED_PARENT_PEFT_TENSOR_COUNT = 506
CERTIFIED_PARENT_LORA_MODULE_COUNT = 253
CERTIFIED_PARENT_CLASSIFIER_TENSOR_COUNT = 64
COMMON_FACETGATE_TARGET_SOURCE_SHA256 = "1c14e1b54846f0ffa1a4a21781703daf7c57ec95a6573a3d2fe0bc14c0f67e3b"

EXPECTED = {
    "rows": 9318,
    "split_counts": {"train": 7454, "validation": 932, "test": 932},
    "labels_sha256": "9cb9afdd88c9a750d3eb6f36aa512dc6ad5b3067aedfac79b138b7fef64a9bb6",
    "ocr_sha256": "eed35bf77f749e98d9222fd571d149b99c4c6a610e29bc5db5783d3489492c0c",
    "split_sha256": "bbcf7e13fc0aab4ac3b5f58a2ebf8880740ed60ad7ecb2cb61910b0497d1b4f0",
    "canonical_row_sha256": "440f8e6982cf4272fc0b6c3e3e6fc9b079a02ce01cd5350a02be0d4a9fa1cf39",
    "ordered_test_sha256": "5f6ac881e9026d682f2f41d65ed1f08e1861d8c8f030bae04d98f9b9db6419db",
    "image_bundle_sha256": "9e134f26c63c23ba871c954ee1d7e03499a47ac3f9bd49d12e0c5ffa977665af",
}

BINARY_TASKS = ["offensive","vulgar","abuse","sarcastic"]
LANGUAGE_LABELS = ["Bangla","English","Banglish"]
TARGET_LABELS = ["Individual","Gender","Political","Social","National","Religious","Others"]
TASK_INSTRUCTION = (
    "Encode this meme for six independent annotations: offensive, vulgar, abuse, sarcastic, "
    "language (Bangla/English/Banglish), and zero or more of seven real target groups. "
    "Do not impose logical implications between tasks. "
)

WORKING = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd()
INPUT_ROOT = Path("/kaggle/input") if Path("/kaggle/input").exists() else Path("/mnt/data")
OUTPUT_ROOT = WORKING / f"M3Offend_9318_MiniCPM_V45_FacetGate_v1_0_1_seed{RUN_SEED}"
ARTIFACT_DIR = OUTPUT_ROOT / "artifacts"
CHECKPOINT_DIR = OUTPUT_ROOT / "checkpoints"
CACHE_DIR = OUTPUT_ROOT / "feature_cache"
LOG_DIR = OUTPUT_ROOT / "logs"
for d in (OUTPUT_ROOT, ARTIFACT_DIR, CHECKPOINT_DIR, CACHE_DIR, LOG_DIR): d.mkdir(parents=True, exist_ok=True)

print({
    "architecture": ARCHITECTURE_NAME,
    "internal_core5_route": "Gated",
    "internal_target_route": "SHTE",
    "model": MODEL_ID,
    "revision": MODEL_REVISION,
    "seed": RUN_SEED,
    "protocol": f"FacetGate v{PROTOCOL_VERSION}",
    "protocol_sha256": PROTOCOL_SHA256,
    "locked_TEST_forwards": 0,
})

In [ ]:
# ================================================================
# 2) STARTUP ORCHESTRATION — CPU ONLY, BEFORE ANY BACKBONE FORWARD
# ================================================================
import os, io, json, re, hashlib, zipfile, shutil, traceback
from pathlib import Path

try:
    import torch
except Exception as e:
    raise RuntimeError("PyTorch must be available before startup provenance inspection.") from e

def sha256_file(path, chunk=8*1024*1024):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        while True:
            b=f.read(chunk)
            if not b: break
            h.update(b)
    return h.hexdigest()

def atomic_json(path, obj):
    path=Path(path); tmp=path.with_suffix(path.suffix+'.tmp')
    tmp.write_text(json.dumps(obj, indent=2, ensure_ascii=False, default=str), encoding='utf-8')
    os.replace(tmp,path)

def recursive_contains(obj, needle):
    if isinstance(obj, dict): return any(recursive_contains(v, needle) for v in obj.values())
    if isinstance(obj, (list,tuple)): return any(recursive_contains(v, needle) for v in obj)
    return str(needle) in str(obj)

def flatten_tensors(obj, prefix=""):
    out={}
    if torch.is_tensor(obj):
        out[prefix.rstrip('.')] = obj
    elif isinstance(obj, dict):
        for k,v in obj.items():
            out.update(flatten_tensors(v, f"{prefix}{k}."))
    return out

def materialize_candidate(seed_root, step):
    direct = seed_root / "checkpoints" / f"candidate_step_{step}.pt"
    if direct.exists(): return direct, {"source":"direct"}
    matches=[]
    for z in seed_root.parent.rglob("*.zip"):
        try:
            with zipfile.ZipFile(z) as zz:
                member=f"checkpoints/candidate_step_{step}.pt"
                if member in zz.namelist(): matches.append((z,member))
        except Exception: pass
    if not matches: return None, {"source":"not_found"}
    uniq={sha256_file(z): (z,m) for z,m in matches}
    if len(uniq)!=1:
        return None, {"source":"ambiguous_zip", "archives":[str(x[0]) for x in matches]}
    z,m=next(iter(uniq.values()))
    out=OUTPUT_ROOT / "gated_parent_materialized" / f"candidate_step_{step}.pt"
    out.parent.mkdir(parents=True,exist_ok=True)
    with zipfile.ZipFile(z) as zz, zz.open(m) as src, open(out,'wb') as dst:
        shutil.copyfileobj(src,dst)
    return out, {"source":"zip", "archive":str(z), "member":m}


def _resolve_gated_head_block(state):
    """Resolve the certified internal-Gated head block without weakening provenance.

    Historical/certified MiniCPM Frozen Common Gated v1.1 snapshots serialize the
    classifier as top-level ``classifier``. FacetGate-native simultaneous snapshots
    serialize the same internal-Gated state as top-level ``discriminative``.
    Exactly one representation must be present.
    """
    present=[k for k in ("classifier","discriminative") if isinstance(state.get(k),dict)]
    if len(present)!=1:
        return None,None,{"reason":"candidate_requires_exactly_one_classifier_or_discriminative_block","present":present,"top_level_keys":list(state)[:30]}
    k=present[0]
    return k,state[k],None

def _has_suffix(flat, suffix):
    return any(k==suffix or k.endswith("."+suffix) for k in flat)

def _minicpm_classifier_expected_keys():
    keys=["missing_vision","missing_text"]
    for stem in ("vision_projection","text_projection"):
        for layer in (0,1):
            keys += [f"{stem}.{layer}.weight",f"{stem}.{layer}.bias"]
    keys += [
        "modality_gate.0.weight","modality_gate.0.bias",
        "shared.0.weight","shared.0.bias","shared.1.weight","shared.1.bias",
    ]
    for task in [*BINARY_TASKS,"language","target"]:
        for part in ("norm","fc1","fc2"):
            keys += [f"task_adapters.{task}.{part}.weight",f"task_adapters.{task}.{part}.bias"]
    for task in BINARY_TASKS:
        keys += [f"binary_heads.{task}.weight",f"binary_heads.{task}.bias"]
    keys += ["language_head.weight","language_head.bias","target_head.weight","target_head.bias"]
    assert len(keys)==64
    return keys

MINICPM_CLASSIFIER_EXPECTED_KEYS=_minicpm_classifier_expected_keys()

def inspect_native_candidate(path, expected_step):
    """CPU-only exact schema audit of a certified MiniCPM Gated candidate.

    MiniCPM's completed Frozen Common Gated v1.1 parent is not reconstructed with
    Qwen/Pali/Intern/Llama adapter names.  Its serialized task adapters are exactly:
      task_adapters.<task>.norm.{weight,bias}
      task_adapters.<task>.fc1.{weight,bias}
      task_adapters.<task>.fc2.{weight,bias}
    """
    state=torch.load(path,map_location='cpu',weights_only=False)
    if not isinstance(state,dict):
        return False,{"reason":"candidate_root_not_dict"}
    peft_block=state.get("peft")
    head_key,head_block,head_err=_resolve_gated_head_block(state)
    if peft_block is None or head_err is not None:
        return False,head_err or {"reason":"candidate_missing_peft_block","top_level_keys":list(state)[:30]}

    peft_flat=flatten_tensors(peft_block)
    head_flat=flatten_tensors(head_block)
    lora=[k for k in peft_flat if "lora_" in k.lower()]
    lora_a=[k for k in lora if "lora_a" in k.lower()]
    missing=[k for k in MINICPM_CLASSIFIER_EXPECTED_KEYS if not _has_suffix(head_flat,k)]
    forbidden=[k for k in head_flat if ".block." in k or ".net." in k]

    if len(peft_flat)!=CERTIFIED_PARENT_PEFT_TENSOR_COUNT:
        return False,{"reason":"certified_peft_tensor_count_mismatch","observed":len(peft_flat),"expected":CERTIFIED_PARENT_PEFT_TENSOR_COUNT}
    if len(lora)!=CERTIFIED_PARENT_PEFT_TENSOR_COUNT or len(lora_a)!=CERTIFIED_PARENT_LORA_MODULE_COUNT:
        return False,{"reason":"certified_lora_inventory_mismatch","lora_tensors":len(lora),"lora_A_modules":len(lora_a),
                      "expected_tensors":CERTIFIED_PARENT_PEFT_TENSOR_COUNT,"expected_modules":CERTIFIED_PARENT_LORA_MODULE_COUNT}
    if len(head_flat)!=CERTIFIED_PARENT_CLASSIFIER_TENSOR_COUNT:
        return False,{"reason":"certified_classifier_tensor_count_mismatch","observed":len(head_flat),"expected":CERTIFIED_PARENT_CLASSIFIER_TENSOR_COUNT,
                      "sample":list(head_flat)[:40]}
    if missing:
        return False,{"reason":"minicpm_parent_classifier_schema_mismatch","adapter_serialization_expected":"norm_fc1_fc2",
                      "missing":missing[:30],"sample":list(head_flat)[:40]}
    if forbidden:
        return False,{"reason":"foreign_backbone_adapter_serialization_detected","forbidden_sample":forbidden[:20],
                      "expected":"norm_fc1_fc2"}

    step=state.get("step",state.get("global_step",expected_step))
    if int(step)!=int(expected_step):
        return False,{"reason":"candidate_step_mismatch","candidate_step":step,"expected":expected_step}
    return True,{
        "top_level_keys":list(state)[:30],
        "gated_head_block_key":head_key,
        "peft_tensor_count":len(peft_flat),
        "lora_tensor_count":len(lora),
        "lora_module_count":len(lora_a),
        "gated_head_tensor_count":len(head_flat),
        "adapter_serialization":"norm_fc1_fc2",
        "gated_head_sample":list(head_flat)[:30],
    }

def normalized_thresholds(marker):
    th=marker.get("validation_thresholds")
    if not isinstance(th,dict): return None
    needed=BINARY_TASKS + [f"target::{x}" for x in TARGET_LABELS]
    if not all(k in th for k in needed): return None
    return {k:float(th[k]) for k in needed}

print("[FacetGate STARTUP] searching for exact same-seed Frozen Common Gated v1.1 parent ...", flush=True)
all_markers=list(INPUT_ROOT.rglob("DEVELOPMENT_COMPLETE.json"))
records=[]
for marker_path in all_markers:
    try:
        marker=json.loads(marker_path.read_text(encoding='utf-8'))
    except Exception:
        continue
    # Only same-seed MiniCPM evidence is an intended parent candidate.
    seed_val=marker.get("seed")
    model_val=marker.get("model_id")
    if str(seed_val)!=str(RUN_SEED): continue
    if model_val and str(model_val)!=MODEL_ID: continue
    rec={"source":str(marker_path),"state":"conflict"}
    try:
        if str(marker.get("model_id")) != MODEL_ID: raise RuntimeError("model_id_mismatch")
        if str(marker.get("model_revision")) != MODEL_REVISION: raise RuntimeError("model_revision_mismatch")
        if str(marker.get("protocol_sha256")) != PARENT_GATED_PROTOCOL_SHA256: raise RuntimeError("parent_protocol_sha256_mismatch")
        if int(marker.get("seed")) != RUN_SEED: raise RuntimeError("seed_mismatch")
        if int(marker.get("test_forward_count",-1)) != 0: raise RuntimeError("development_TEST_forward_count_not_zero")
        if str(marker.get("binding_sha256")) != MINICPM_NATIVE_BINDING_SHA256:
            raise RuntimeError("minicpm_native_binding_sha256_mismatch")
        step=int(marker.get("best_step"))
        cert=CERTIFIED_PARENT[RUN_SEED]
        if step!=int(cert["step"]):
            raise RuntimeError(f"certified_parent_best_step_mismatch:observed={step}:expected={cert['step']}")
        thresholds=normalized_thresholds(marker)
        if thresholds is None: raise RuntimeError("certified_validation_thresholds_missing_or_incomplete")
        seed_root=marker_path.parent.parent
        run_cfg=seed_root/"artifacts"/"run_config.json"
        binding=seed_root/"artifacts"/"minicpm_native_binding.json"
        if not run_cfg.exists(): raise RuntimeError("parent_run_config_missing")
        if not binding.exists(): raise RuntimeError("parent_minicpm_native_binding_missing")
        joined=run_cfg.read_text(encoding='utf-8',errors='ignore') + "\n" + binding.read_text(encoding='utf-8',errors='ignore')
        for required in (EXPECTED["labels_sha256"],EXPECTED["ocr_sha256"],EXPECTED["split_sha256"]):
            if required not in joined: raise RuntimeError(f"dataset_binding_missing:{required[:12]}")
        ckpt,source=materialize_candidate(seed_root,step)
        if ckpt is None: raise RuntimeError(f"selected_checkpoint_not_found:{source}")
        ck_sha=sha256_file(ckpt)
        if ck_sha!=CERTIFIED_PARENT[RUN_SEED]["checkpoint_sha256"]:
            raise RuntimeError(f"certified_parent_checkpoint_sha256_mismatch:{ck_sha}")
        ok,ckinfo=inspect_native_candidate(ckpt,step)
        if not ok: raise RuntimeError(f"selected_checkpoint_invalid:{ckinfo}")
        rec.update({
            "state":"valid","seed_root":str(seed_root),"step":step,"checkpoint":str(ckpt),
            "checkpoint_sha256":ck_sha,"checkpoint_source":source,
            "thresholds":thresholds,"checkpoint_inspection":ckinfo,
            "test_forward_count":0,
            "marker_sha256":sha256_file(marker_path),
            "binding_sha256":str(marker.get("binding_sha256")),
            "parent_run_config":str(run_cfg),
            "parent_native_binding":str(binding),
            "validation_raw_at_0_5":marker.get("validation_raw_at_0_5"),
            "validation_calibrated":marker.get("validation_calibrated"),
        })
    except Exception as e:
        rec["reason"]=str(e)
    records.append(rec)

valid=[r for r in records if r["state"]=="valid"]
conflicts=[r for r in records if r["state"]=="conflict"]
# Deduplicate byte-identical valid candidates.
valid_by_sha={r["checkpoint_sha256"]:r for r in valid}
if len(valid_by_sha)==1 and not conflicts:
    GATED_PARENT=next(iter(valid_by_sha.values()))
    EXECUTION_MODE="REUSE_EXISTING_GATED"
elif len(valid_by_sha)==0 and len(conflicts)==0:
    GATED_PARENT=None
    EXECUTION_MODE="SIMULTANEOUS_FROM_SCRATCH"
else:
    GATED_PARENT=None
    EXECUTION_MODE="FAIL_CLOSED"

STARTUP={
    "architecture":"FacetGate",
    "seed":RUN_SEED,
    "mode":EXECUTION_MODE,
    "records":records,
    "valid_unique_count":len(valid_by_sha),
    "conflict_count":len(conflicts),
    "selected":GATED_PARENT,
    "test_forward_count":0,
}
atomic_json(ARTIFACT_DIR/"FACETGATE_STARTUP_DISCOVERY.json",STARTUP)
print(json.dumps(STARTUP,indent=2),flush=True)
if EXECUTION_MODE=="FAIL_CLOSED":
    raise RuntimeError("FacetGate startup FAIL_CLOSED: same-seed MiniCPM parent evidence is incomplete, mismatched, or ambiguous. Inspect FACETGATE_STARTUP_DISCOVERY.json.")
print(f"[FacetGate ROUTE] {EXECUTION_MODE} | TEST forwards=0",flush=True)

In [ ]:
# ================================================================
# 3) EMBEDDED ROUTE-SPECIFIC CHILD NOTEBOOKS — R18
#    Do not edit these payloads. The controller materializes exactly one.
# ================================================================
import base64, zlib, hashlib

TARGET_ONLY_CHILD_ZLIB_B64 = '...'
SIMULTANEOUS_CHILD_ZLIB_B64 = '...'

TARGET_ONLY_TEMPLATE_SHA256 = 'e84cd4cd50497c8ec693bd82f42df338e329b67b55a0dc2c62e855d7b14b04f8'
SIMULTANEOUS_TEMPLATE_SHA256 = '1df0f4144f9445e3622030e145e1870878cb90df4fb3518b1f21b5256bb11676'

def _decode_child(blob_b64, expected_sha):
    raw=zlib.decompress(base64.b64decode(blob_b64.encode("ascii")))
    got=hashlib.sha256(raw).hexdigest()
    if got!=expected_sha:
        raise RuntimeError(f"Embedded child SHA mismatch: {got} != {expected_sha}")
    return raw

# CPU-only integrity check; still no model/backbone load or forward.
_decode_child(TARGET_ONLY_CHILD_ZLIB_B64,TARGET_ONLY_TEMPLATE_SHA256)
_decode_child(SIMULTANEOUS_CHILD_ZLIB_B64,SIMULTANEOUS_TEMPLATE_SHA256)
print({
    "embedded_children":"PASS",
    "target_only_template_sha256":TARGET_ONLY_TEMPLATE_SHA256,
    "simultaneous_template_sha256":SIMULTANEOUS_TEMPLATE_SHA256,
    "selected_mode":EXECUTION_MODE,
    "TEST_forwards":0,
})


In [ ]:
# ================================================================
# 4) RESUME EXACT CURRENT PROGRESS OR MATERIALIZE ONE R18 CHILD
# ================================================================
import json, hashlib, shutil
from pathlib import Path

RESUMED_FROM_PROGRESS=False
RESUME_SOURCE_ROOT=None
CHILD_NOTEBOOK=None
EXECUTED_NOTEBOOK=None

def _resume_sha256(path, chunk=8*1024*1024):
    h=hashlib.sha256()
    with open(path,"rb") as f:
        while True:
            b=f.read(chunk)
            if not b: break
            h.update(b)
    return h.hexdigest()

def _inspect_finalized_progress(score_path):
    score_path=Path(score_path)
    if score_path.parent.name!="artifacts":
        return None,"scorecard_not_under_artifacts"
    root=score_path.parent.parent
    art=root/"artifacts"
    bind_path=art/"FACETGATE_FINAL_COMPOSITE_BINDING.json"
    thr_path=art/"FACETGATE_THRESHOLDS.json"
    counter_path=art/"TEST_FORWARD_COUNTER.json"
    compat_path=art/"SHTE_COMPATIBILITY_REPLAY.json"
    for p in (bind_path,thr_path,counter_path,compat_path):
        if not p.exists():
            return None,f"missing_finalization_artifact:{p.name}"
    try:
        score=json.loads(score_path.read_text(encoding="utf-8"))
        bind=json.loads(bind_path.read_text(encoding="utf-8"))
        counter=json.loads(counter_path.read_text(encoding="utf-8"))
        compat=json.loads(compat_path.read_text(encoding="utf-8"))
    except Exception as e:
        return None,f"json_parse_failed:{type(e).__name__}"

    # Exact scientific/provenance binding; no result is reused across seed/model/revision/protocol.
    if score.get("architecture")!="FacetGate" or bind.get("architecture")!="FacetGate":
        return None,"architecture_mismatch"
    if int(score.get("seed",-1))!=RUN_SEED or int(bind.get("seed",-1))!=RUN_SEED:
        return None,"seed_mismatch"
    if bind.get("model_id")!=MODEL_ID or bind.get("model_revision")!=MODEL_REVISION:
        return None,"model_revision_mismatch"
    if bind.get("protocol_sha256")!=PROTOCOL_SHA256:
        return None,"protocol_sha_mismatch"
    if score.get("execution_mode")!=EXECUTION_MODE or bind.get("execution_mode")!=EXECUTION_MODE:
        return None,"execution_mode_mismatch"
    if int(score.get("locked_TEST_forwards",-1))!=0 or int(bind.get("locked_TEST_forwards",-1))!=0 or int(counter.get("TEST_forwards",-1))!=0:
        return None,"TEST_forward_counter_nonzero"
    if not bool(score.get("core5_identity",{}).get("pass",False)):
        return None,"core5_identity_failed"

    # Same exact Gated parent/c* must be bound when this is the reuse family.
    if EXECUTION_MODE=="REUSE_EXISTING_GATED":
        if GATED_PARENT is None:
            return None,"reuse_mode_without_current_parent"
        if int(score.get("selected_internal_gated_step",-1))!=int(GATED_PARENT["step"]):
            return None,"selected_gated_step_mismatch"
        bstep=bind.get("selected_internal_gated_step")
        if bstep is not None and int(bstep)!=int(GATED_PARENT["step"]):
            return None,"binding_gated_step_mismatch"
        bsha=bind.get("selected_internal_gated_sha256")
        if bsha and str(bsha)!=str(GATED_PARENT["checkpoint_sha256"]):
            return None,"binding_gated_sha_mismatch"

    step=int(score.get("selected_internal_shte_step",-1))
    if step not in [200,400,600,800,932]:
        return None,"selected_shte_step_invalid"
    if int(bind.get("selected_internal_shte_step",-2))!=step:
        return None,"selected_shte_step_binding_mismatch"

    # Frozen compatibility replay must contain all five candidates exactly once.
    if not isinstance(compat,list):
        return None,"compatibility_replay_not_list"
    csteps=sorted(int(x.get("step",-1)) for x in compat if isinstance(x,dict))
    if csteps!=[200,400,600,800,932]:
        return None,f"compatibility_replay_steps_mismatch:{csteps}"

    ck=root/"checkpoints"/f"facetgate_internal_shte_candidate_step_{step}.pt"
    if not ck.exists():
        return None,f"selected_shte_checkpoint_missing:{ck}"
    actual_ck_sha=_resume_sha256(ck)
    expected_ck_sha=bind.get("selected_internal_shte_sha256")
    if expected_ck_sha and str(expected_ck_sha)!=actual_ck_sha:
        return None,"selected_shte_sha_mismatch"

    # Check the selected specialist payload itself, not just its file name/hash.
    try:
        st=torch.load(ck,map_location="cpu",weights_only=False)
    except Exception as e:
        return None,f"selected_shte_load_failed:{type(e).__name__}"
    if not isinstance(st,dict) or int(st.get("step",-1))!=step or not isinstance(st.get("state_dict"),dict):
        return None,"selected_shte_payload_schema_mismatch"
    if st.get("protocol_sha256") not in (None,PROTOCOL_SHA256):
        return None,"selected_shte_payload_protocol_mismatch"

    return {
        "root":root,"art":art,"score":score,"score_path":score_path,"bind":bind,
        "selected_step":step,"selected_ckpt":ck,
        "selected_ckpt_sha":actual_ck_sha,"score_sha":_resume_sha256(score_path),
    },None

resume_records=[]
if RESUME_FROM_INPUT_PROGRESS and INPUT_ROOT.exists():
    for score_path in INPUT_ROOT.rglob("FACETGATE_VALIDATION_SCORECARD.json"):
        cand,reason=_inspect_finalized_progress(score_path)
        # Nonmatching seeds are normal when multiple prior runs are mounted.
        if cand is not None:
            resume_records.append({"scorecard":str(score_path),"state":"valid","candidate":cand})
        elif reason!="seed_mismatch":
            resume_records.append({"scorecard":str(score_path),"state":"rejected","reason":reason})

valid=[r["candidate"] for r in resume_records if r["state"]=="valid"]
uniq={(c["score_sha"],c["selected_ckpt_sha"]):c for c in valid}
valid_unique=list(uniq.values())
if len(valid_unique)>1:
    raise RuntimeError(
        "Conflicting finalized same-seed FacetGate progress datasets found; refusing to choose one: "+
        json.dumps([{"root":str(c["root"]),"score_sha":c["score_sha"],"selected_ckpt_sha":c["selected_ckpt_sha"]} for c in valid_unique],indent=2)
    )

if len(valid_unique)==1:
    c=valid_unique[0]
    RESUME_SOURCE_ROOT=str(c["root"])
    # Copy exact final scientific artifacts + selected SHTE checkpoint only.
    for p in c["art"].rglob("*"):
        if p.is_file():
            rel=p.relative_to(c["art"])
            dst=ARTIFACT_DIR/rel
            dst.parent.mkdir(parents=True,exist_ok=True)
            shutil.copy2(p,dst)
    dst_ck=CHECKPOINT_DIR/c["selected_ckpt"].name
    shutil.copy2(c["selected_ckpt"],dst_ck)

    # Rewrite only location-dependent binding paths. Scientific values/hashes remain exact.
    bind_path=ARTIFACT_DIR/"FACETGATE_FINAL_COMPOSITE_BINDING.json"
    bind=json.loads(bind_path.read_text(encoding="utf-8"))
    bind["selected_internal_shte_checkpoint"]=str(dst_ck)
    bind["selected_internal_shte_sha256"]=_resume_sha256(dst_ck)
    if GATED_PARENT is not None:
        bind["selected_internal_gated_checkpoint"]=str(GATED_PARENT["checkpoint"])
        bind["selected_internal_gated_sha256"]=GATED_PARENT["checkpoint_sha256"]
    atomic_json(bind_path,bind)

    resume_marker={
        "status":"RESUMED_FINALIZED_DEVELOPMENT",
        "architecture":"FacetGate",
        "seed":RUN_SEED,
        "model_id":MODEL_ID,
        "model_revision":MODEL_REVISION,
        "protocol_sha256":PROTOCOL_SHA256,
        "execution_mode":EXECUTION_MODE,
        "source_progress_root":RESUME_SOURCE_ROOT,
        "source_scorecard_sha256":c["score_sha"],
        "selected_internal_shte_step":c["selected_step"],
        "selected_internal_shte_sha256":c["selected_ckpt_sha"],
        "locked_TEST_forwards":0,
        "gpu_worker_reexecuted":False,
        "feature_cache_rebuilt":False,
        "training_repeated":False,
    }
    atomic_json(ARTIFACT_DIR/"FACETGATE_RESUME_PROVENANCE.json",resume_marker)
    atomic_json(ARTIFACT_DIR/"FACETGATE_RESUME_DISCOVERY.json",{"records":resume_records,"selected":resume_marker})
    RESUMED_FROM_PROGRESS=True
    print("[FacetGate RESUME] exact finalized same-seed progress verified; expensive child/GPU execution will be skipped.")
    print(json.dumps(resume_marker,indent=2))
else:
    atomic_json(ARTIFACT_DIR/"FACETGATE_RESUME_DISCOVERY.json",{"records":resume_records,"selected":None})

if not RESUMED_FROM_PROGRESS:
    # ================================================================
    # 4) MATERIALIZE EXACTLY ONE CHILD + STATIC SCIENTIFIC QA
    # ================================================================
    import nbformat, hashlib, ast, re, symtable, builtins
    from pathlib import Path

    if EXECUTION_MODE not in {"REUSE_EXISTING_GATED","SIMULTANEOUS_FROM_SCRATCH"}:
        raise RuntimeError(f"Unexpected route at materialization: {EXECUTION_MODE}")

    if EXECUTION_MODE=="REUSE_EXISTING_GATED":
        raw=_decode_child(TARGET_ONLY_CHILD_ZLIB_B64,TARGET_ONLY_TEMPLATE_SHA256)
        route_tag="reuse_parent"
    else:
        raw=_decode_child(SIMULTANEOUS_CHILD_ZLIB_B64,SIMULTANEOUS_TEMPLATE_SHA256)
        route_tag="simultaneous"

    text=raw.decode("utf-8")
    replacements={
        "__RUN_SEED__":str(int(RUN_SEED)),
        "__EXECUTION_MODE__":EXECUTION_MODE,
        "__GATED_PARENT_REPR__":repr(GATED_PARENT),
    }
    for old,new in replacements.items():
        if old not in text:
            raise RuntimeError(f"Required child placeholder absent before replacement: {old}")
        text=text.replace(old,new)

    for forbidden in ("__RUN_SEED__","__EXECUTION_MODE__","__GATED_PARENT_REPR__","__EXPECTED_CHILD_MODE__"):
        if forbidden in text:
            raise RuntimeError(f"Unresolved child placeholder remains: {forbidden}")

    child=nbformat.reads(text,as_version=4)
    nbformat.validate(child)

    # Controller -> child binding verification.
    _binding_tree = ast.parse(child.cells[1].source)
    _materialized_mode = None
    for _node in _binding_tree.body:
        if isinstance(_node, ast.Assign) and any(
            isinstance(_t, ast.Name) and _t.id == "EXECUTION_MODE" for _t in _node.targets
        ):
            _materialized_mode = ast.literal_eval(_node.value)
            break
    if _materialized_mode != EXECUTION_MODE:
        raise RuntimeError(
            f"Controller/child EXECUTION_MODE binding corruption: "
            f"{_materialized_mode!r} != {EXECUTION_MODE!r}"
        )

    # Clear all stale outputs/execution counters, then syntax-compile every code cell.
    for i,c in enumerate(child.cells):
        if c.cell_type=="code":
            c.outputs=[]
            c.execution_count=None
            compile(c.source,f"<child_cell_{i}>","exec")

    # Fresh-kernel undefined-global audit across ALL materialized child code cells.
    # This catches missing imports before any child execution/GPU work.
    def _undefined_global_names(src):
        st=symtable.symtable(src,"<materialized_child>","exec")
        module_defs={
            s.get_name() for s in st.get_symbols()
            if s.is_assigned() or s.is_imported() or s.is_namespace() or s.is_parameter()
        }
        refs=set()
        def walk(tab):
            for s in tab.get_symbols():
                if s.is_referenced() and (s.is_global() or tab.get_type()=="module"):
                    refs.add(s.get_name())
            for ch in tab.get_children():
                walk(ch)
        walk(st)
        return sorted(refs-module_defs-set(dir(builtins)))

    _child_script="\n\n".join(c.source for c in child.cells if c.cell_type=="code")
    _undef_child=_undefined_global_names(_child_script)
    if _undef_child:
        raise RuntimeError(f"Materialized child has undefined global names: {_undef_child}")

    child_text="\n".join(c.source for c in child.cells)

    # Route-independent frozen-contract assertions.
    required_fragments=[
        'ARCHITECTURE_NAME = "FacetGate"',
        'PROTOCOL_SHA256 = "37b7d30a994fb98225d49d76cca5768a838d456abf844dc1a5d93019a0cd15aa"',
        'PARENT_GATED_PROTOCOL_SHA256 = "627bdf5a77ab694581ff590c32f8885c901ebad09e8df86dbab07762caaf95dd"',
        'class MiniCPMGatedAdapter',
        'self.norm=nn.LayerNorm',
        'self.fc1=nn.Linear',
        'self.fc2=nn.Linear',
        'class ResidualFacetAdapter',
        'class FacetGateTarget',
        'COMMON_FACETGATE_TARGET_SOURCE_SHA256',
        'CANDIDATES={200,400,600,800,932}',
        'test_forward_count',
        'zG.detach()',
    ]
    missing=[s for s in required_fragments if s not in child_text]
    if missing:
        raise RuntimeError(f"Selected child missing frozen-contract fragments: {missing}")

    # The MiniCPM parent/internal-Gated adapter MUST use norm/fc1/fc2, never the foreign
    # Qwen/Pali/Intern/Llama serialized adapter names that caused the previous failure.
    worker_cell=child.cells[4].source
    tree=ast.parse(worker_cell)
    worker_source=None
    for node in tree.body:
        if isinstance(node,ast.Assign) and any(isinstance(t,ast.Name) and t.id=="WORKER_SOURCE" for t in node.targets):
            worker_source=ast.literal_eval(node.value)
            break
    if not isinstance(worker_source,str):
        raise RuntimeError("Could not recover embedded MiniCPM worker source")
    compile(worker_source,"<minicpm_facetgate_worker>","exec")
    if "dist.destroy_process_group()" not in worker_source:
        raise RuntimeError("R18 worker missing explicit NCCL teardown before CPU bootstrap")
    if worker_source.index("dist.destroy_process_group()") > worker_source.index("rng=np.random.default_rng"):
        raise RuntimeError("R18 NCCL teardown occurs after long bootstrap; watchdog risk reintroduced")
    if "dist.barrier();dist.destroy_process_group()" in worker_source:
        raise RuntimeError("R16 late teardown tail reintroduced")
    _undef_worker=_undefined_global_names(worker_source)
    if _undef_worker:
        raise RuntimeError(f"Embedded MiniCPM FacetGate worker has undefined global names: {_undef_worker}")

    def _class_source(src,name):
        tr=ast.parse(src)
        for n in tr.body:
            if isinstance(n,ast.ClassDef) and n.name==name:
                lines=src.splitlines()
                return "\n".join(lines[n.lineno-1:n.end_lineno])
        raise RuntimeError(f"Worker class missing: {name}")

    parent_adapter_src=_class_source(worker_source,"MiniCPMGatedAdapter")
    if ".block" in parent_adapter_src or ".net" in parent_adapter_src:
        raise RuntimeError("Foreign adapter serialization leaked into MiniCPM parent adapter")
    for frag in ("self.norm","self.fc1","self.fc2"):
        if frag not in parent_adapter_src:
            raise RuntimeError(f"MiniCPM parent adapter missing {frag}")

    # Exact common Target specialist source identity used by the four successful reference notebooks.
    target_src=_class_source(worker_source,"ResidualFacetAdapter")+"\n\n"+_class_source(worker_source,"FacetGateTarget")
    target_src_sha=hashlib.sha256(target_src.encode("utf-8")).hexdigest()
    if target_src_sha!=COMMON_FACETGATE_TARGET_SOURCE_SHA256:
        raise RuntimeError(
            f"Common FacetGate Target specialist source drift: {target_src_sha} "
            f"!= {COMMON_FACETGATE_TARGET_SOURCE_SHA256}"
        )

    # Shared-forward safeguards.
    # IMPORTANT: verify the native embedding call structurally, not by a fragile local
    # variable name.  The MiniCPM worker intentionally binds the native model as `nat`
    # (`nat.get_vllm_embedding(...)`); older controller code incorrectly demanded the
    # literal string `native.get_vllm_embedding`, which caused a false preflight failure.
    _worker_ast = ast.parse(worker_source)
    _has_get_vllm_embedding_call = any(
        isinstance(n, ast.Call)
        and isinstance(n.func, ast.Attribute)
        and n.func.attr == "get_vllm_embedding"
        for n in ast.walk(_worker_ast)
    )
    if not _has_get_vllm_embedding_call:
        raise RuntimeError("Worker missing MiniCPM native get_vllm_embedding(...) call")

    for frag in (
        'v.detach()',
        't.detach()',
        'J.detach()',
        'zG.detach()',
        'len(targets)!=253',
    ):
        if frag not in worker_source:
            raise RuntimeError(f"Worker missing shared-forward/frozen binding fragment: {frag}")

    # Exact supplied successful MiniCPM Gated quantization boundary.
    # Vision LoRA is forbidden, but a frozen quantized VPM is allowed; the parent only
    # explicitly excludes llm.lm_head and resampler.attn.out_proj from BnB conversion.
    if 'BNB_4BIT_SKIP_MODULES = ["llm.lm_head","resampler.attn.out_proj"]' not in child_text:
        raise RuntimeError("MiniCPM certified BnB skip-module configuration drift")
    if 'if vpm_4!=0' in worker_source:
        raise RuntimeError("Obsolete false VPM-full-precision guard reintroduced")
    for _qfrag in (
        'llm_int8_skip_modules=list(CFG["bnb_4bit_skip_modules"])',
        'bridge=native.get_submodule("resampler.kv_proj")',
        'lang_probe=native.get_submodule("llm.model.layers.0.self_attn.q_proj")',
        'MiniCPM resampler.kv_proj must remain NF4 Linear4bit before LoRA injection',
        'MiniCPM language probe must remain NF4 Linear4bit before LoRA injection',
        'Vision LoRA forbidden by certified MiniCPM parent binding',
        'Vision trainables detected after PEFT injection',
    ):
        if _qfrag not in worker_source:
            raise RuntimeError(f"MiniCPM supplied-parent quantization binding fragment missing: {_qfrag}")

    # Exact supplied MiniCPM Gated native processor/hidden-state binding.
    for _frag in (
        'marker=CFG["native_image_marker"]',
        'images=[[image]]',
        'bounds=normalize_bounds(batch["image_bound"][0])',
        'exact_content_positions',
        'nat.llm.model(',
        'max_length=CFG["native_max_input_length"]',
        'native_binding_canary()',
    ):
        if _frag not in worker_source:
            raise RuntimeError(f"MiniCPM supplied-parent native binding fragment missing: {_frag}")

    # TARGET_ONLY must carry a real certified parent. SIMULTANEOUS must carry None.
    if EXECUTION_MODE=="REUSE_EXISTING_GATED":
        if GATED_PARENT is None:
            raise RuntimeError("REUSE route selected without GATED_PARENT")
        if "GATED_PARENT = None" in child_text:
            raise RuntimeError("REUSE child unexpectedly materialized without parent")
    else:
        if GATED_PARENT is not None:
            raise RuntimeError("SIMULTANEOUS route selected with non-null parent")

    CHILD_NOTEBOOK=OUTPUT_ROOT/f"MiniCPM_V45_FacetGate_seed{RUN_SEED}_{route_tag}.ipynb"
    nbformat.write(child,CHILD_NOTEBOOK)

    selected_sha=hashlib.sha256(CHILD_NOTEBOOK.read_bytes()).hexdigest()
    qa={
        "status":"PASS",
        "architecture":"FacetGate",
        "seed":RUN_SEED,
        "mode":EXECUTION_MODE,
        "child_notebook":str(CHILD_NOTEBOOK),
        "child_sha256":selected_sha,
        "protocol_sha256":PROTOCOL_SHA256,
        "common_target_source_sha256":target_src_sha,
        "minicpm_parent_adapter_serialization":"norm_fc1_fc2",
        "lora_target_count":253,
        "locked_TEST_forwards":0,
    }
    atomic_json(ARTIFACT_DIR/"CONTROLLER_CHILD_STATIC_QA.json",qa)
    print(json.dumps(qa,indent=2),flush=True)


In [ ]:
# ================================================================
# 5) EXECUTE SELECTED CHILD IN A FRESH KERNEL — UNLESS EXACT PROGRESS RESUMED
# ================================================================
import os, sys, subprocess
from pathlib import Path

if RESUMED_FROM_PROGRESS:
    EXECUTED_NOTEBOOK=None
    print("[FacetGate controller] resume path active; selected child execution skipped.")
else:
    import os, sys, subprocess
    from pathlib import Path

    EXECUTED_NOTEBOOK=CHILD_NOTEBOOK.with_name(CHILD_NOTEBOOK.stem+"_EXECUTED.ipynb")
    cmd=[
        sys.executable,"-m","jupyter","nbconvert",
        "--to","notebook","--execute",
        "--ExecutePreprocessor.timeout=-1",
        "--output",EXECUTED_NOTEBOOK.name,
        str(CHILD_NOTEBOOK),
    ]
    print("[FacetGate controller] executing selected child:", " ".join(cmd), flush=True)
    proc=subprocess.run(cmd,cwd=str(OUTPUT_ROOT),env=os.environ.copy())
    if proc.returncode!=0:
        raise RuntimeError(
            f"Selected FacetGate child failed with return code {proc.returncode}. "
            f"Inspect {CHILD_NOTEBOOK}, {EXECUTED_NOTEBOOK}, and {LOG_DIR}."
        )
    print("[FacetGate controller] selected child completed successfully:",EXECUTED_NOTEBOOK)


In [ ]:
# ================================================================
# 6) FINAL CONTROLLER SUMMARY + DURABLE DEVELOPMENT MARKER
# ================================================================
import json, zipfile
from pathlib import Path

score_path=ARTIFACT_DIR/"FACETGATE_VALIDATION_SCORECARD.json"
if not score_path.exists():
    raise RuntimeError(f"FacetGate run/resume returned without validation scorecard: {score_path}")
score=json.loads(score_path.read_text(encoding="utf-8"))

locked_test=int(score.get("locked_TEST_forwards",score.get("test_forward_count",0)))
if locked_test!=0:
    raise RuntimeError("Development TEST-forward invariant violated")

selected_step=int(score["selected_internal_shte_step"])
selected_ck=CHECKPOINT_DIR/f"facetgate_internal_shte_candidate_step_{selected_step}.pt"
binding=ARTIFACT_DIR/"FACETGATE_FINAL_COMPOSITE_BINDING.json"
thresholds=ARTIFACT_DIR/"FACETGATE_THRESHOLDS.json"
if not selected_ck.exists():
    raise RuntimeError(f"Missing selected SHTE checkpoint: {selected_ck}")
if not binding.exists() or not thresholds.exists():
    raise RuntimeError("Missing final FacetGate binding/threshold artifacts")

summary={
    "architecture":"FacetGate",
    "model":MODEL_ID,
    "revision":MODEL_REVISION,
    "seed":RUN_SEED,
    "mode":EXECUTION_MODE,
    "resumed_from_progress":bool(RESUMED_FROM_PROGRESS),
    "resume_source_root":RESUME_SOURCE_ROOT,
    "selected_child":None if CHILD_NOTEBOOK is None else str(CHILD_NOTEBOOK),
    "executed_child":None if EXECUTED_NOTEBOOK is None else str(EXECUTED_NOTEBOOK),
    "protocol_sha256":PROTOCOL_SHA256,
    "locked_TEST_forwards":0,
    "scorecard":score,
}
atomic_json(ARTIFACT_DIR/"FACETGATE_CONTROLLER_COMPLETE.json",summary)

complete={
    "status":"DEVELOPMENT_COMPLETE_TEST_LOCKED",
    "architecture":"FacetGate",
    "model_id":MODEL_ID,
    "model_revision":MODEL_REVISION,
    "seed":RUN_SEED,
    "protocol_version":PROTOCOL_VERSION,
    "protocol_sha256":PROTOCOL_SHA256,
    "execution_mode":EXECUTION_MODE,
    "selected_internal_gated_step":int(score["selected_internal_gated_step"]),
    "selected_internal_shte_step":selected_step,
    "scorecard_sha256":sha256_file(score_path),
    "selected_internal_shte_sha256":sha256_file(selected_ck),
    "final_composite_binding_sha256":sha256_file(binding),
    "thresholds_sha256":sha256_file(thresholds),
    "locked_TEST_forwards":0,
    "resumed_from_progress":bool(RESUMED_FROM_PROGRESS),
}
atomic_json(ARTIFACT_DIR/"FACETGATE_DEVELOPMENT_COMPLETE.json",complete)

zip_path=OUTPUT_ROOT/f"M3Offend_9318_MiniCPM_V45_FacetGate_seed{RUN_SEED}_DEVELOPMENT_RESULTS.zip"
with zipfile.ZipFile(zip_path,"w",zipfile.ZIP_DEFLATED) as z:
    for p in ARTIFACT_DIR.rglob("*"):
        if p.is_file():
            z.write(p,p.relative_to(OUTPUT_ROOT))
    z.write(selected_ck,selected_ck.relative_to(OUTPUT_ROOT))

print(json.dumps(summary,indent=2), flush=True)
print("Development marker:",json.dumps(complete,indent=2))
print("Results ZIP:",zip_path)
print("NOTE: Locked TEST has not been forwarded, selected, calibrated, or repaired in this notebook.")
